# Data Query Notebkook
The purpose of this notebook is to query https://cmsoms.cern.ch/cms/run_3/index in order to get a list of all runs that satisfy three criteria:
* the run is using protons (```  q.filter("fill_type_runtime", fillType)```)
* collisions from 2026 (```q.filter("l1_hlt_mode", l1_hlt_mode)```)
* there is a stable beam (```q.filter("stable_beam", True)```)

[Here](https://gitlab.cern.ch/cmsoms/oms-api-client/-/tree/master?ref_type=heads#examples) is an example how to use the API

## Set Up

In [3]:
from omsapi import OMSAPI
import json

In [4]:
omsapi = OMSAPI("https://cmsoms.cern.ch/agg/api", "v1", cert_verify=False, verbose = True, throw_on_err=True)

omsapi.auth_krb() # Authorisation for https using kerberos, also supports 2FA

## Fetching all eras

In [11]:
q = omsapi.query("eras") # creates a query object (OMSQUERY)
q.set_verbose(True)
q.paginate(per_page=999)
#q.attrs(["type"])

In [12]:
response = q.data() # execute query and retreive data - returns requests.Response object
d = response.json()

# Display JSON data
print(json.dumps(d, indent=2))

https://cmsoms.cern.ch/agg/api/v1/eras?page[offset]=0&page[limit]=999
{
  "data": [
    {
      "id": "2011A",
      "type": "eras",
      "attributes": {
        "start_time": "2011-03-13T16:01:57Z",
        "start_run": 160401,
        "end_fill": 2040,
        "start_fill": 1613,
        "name": "2011A",
        "end_time": "2011-08-22T16:10:13Z",
        "end_run": 173692
      },
      "relationships": {
        "fills": {
          "links": {
            "self": "https://https://vocms0185.cern.ch/agg/api/v1/eras/2011A/relationships/fills",
            "related": "https://https://vocms0185.cern.ch/agg/api/v1/eras/2011A/fills"
          }
        }
      },
      "links": {
        "self": "https://https://vocms0185.cern.ch/agg/api/v1/eras/2011A"
      }
    },
    {
      "id": "2011B",
      "type": "eras",
      "attributes": {
        "start_time": "2011-09-07T11:28:51Z",
        "start_run": 175828,
        "end_fill": 2267,
        "start_fill": 2083,
        "name": "2011B",

In [7]:
len(d["data"])

100

In [8]:
for era in d["data"]:
    attrs = era["attributes"]
    print(attrs["name"], attrs["start_time"], "->", attrs["end_time"])

2011A 2011-03-13T16:01:57Z -> 2011-08-22T16:10:13Z
2011B 2011-09-07T11:28:51Z -> 2011-10-30T16:31:31Z
2011HI 2011-11-12T04:50:18Z -> 2011-12-07T09:25:51Z
2012A 2012-04-05T07:56:28Z -> 2012-05-08T07:20:06Z
2012B 2012-05-10T03:50:22Z -> 2012-06-18T11:10:14Z
2012C 2012-07-01T23:04:16Z -> 2012-09-27T09:57:48Z
2012D 2012-09-28T05:22:35Z -> 2012-12-16T22:08:59Z
2013HI 2013-01-20T09:56:16Z -> 2013-02-10T05:11:51Z
2013A 2013-02-11T16:33:54Z -> 2013-02-14T06:26:26Z
2015A 2015-06-09T20:17:04Z -> 2015-07-06T00:55:16Z
2015B 2015-07-06T09:04:03Z -> 2015-08-07T06:33:14Z
2015C 2015-08-07T12:15:11Z -> 2015-09-15T03:17:05Z
2015D 2015-09-16T10:08:26Z -> 2015-11-03T07:16:10Z
2016A 2016-04-22T21:41:30Z -> 2016-04-27T10:19:54Z
2016B 2016-04-28T20:33:55Z -> 2016-06-21T07:02:01Z
2016C 2016-06-24T00:50:03Z -> 2016-07-04T08:18:15Z
2016D 2016-07-04T08:33:45Z -> 2016-07-15T04:56:16Z
2016E 2016-07-15T05:43:36Z -> 2016-07-25T21:28:13Z
2016F 2016-07-29T20:12:14Z -> 2016-08-14T07:05:15Z
2016G 2016-08-14T07:33:30Z ->

We are only intersten in proton-proton collisions, so we discard HI (heavy ion) collisions!

## Getting the fill for each run era
Each LHC Run Era contains multiple fills. We want to list them here

In [26]:
q = omsapi.query("fills")
response = q.data() # execute query and retreive data - returns requests.Response object
d = response.json()["data"][0:3]

# Display JSON data
print(json.dumps(d, indent=2))

https://cmsoms.cern.ch/agg/api/v1/fills?page[offset]=0&page[limit]=10
[
  {
    "id": "0",
    "type": "fills",
    "attributes": {
      "peak_lumi": null,
      "bunches_colliding": null,
      "to_tracker_ready_time": null,
      "intensity_beam2": null,
      "to_ready_time": null,
      "intensity_beam1": null,
      "crossing_angle": null,
      "dump_ready_to_dump_time": null,
      "end_stable_beam": null,
      "duration": null,
      "b_field": null,
      "stop_crossing_angle": null,
      "beta_star": null,
      "init_lumi": null,
      "era": null,
      "delivered_lumi_stablebeams": 0.0,
      "peak_specific_lumi": null,
      "bunches_target": null,
      "injection_scheme": null,
      "stable_beams": false,
      "recorded_lumi": null,
      "delivered_lumi": null,
      "efficiency_lumi_stablebeams": null,
      "last_run_number": null,
      "energy": null,
      "fill_number": 0,
      "start_crossing_angle": null,
      "recorded_lumi_stablebeams": 0.0,
      "eff

In [25]:
len(d["data"])

TypeError: list indices must be integers or slices, not str

In [37]:
from oms_pipeline import connect, _fetch_all
print([e["attributes"]["name"] for e in _fetch_all(connect(), "eras")])

ModuleNotFoundError: No module named 'oms_pipeline'